# 1) Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 29
TARGET_COLUMN = "attack_detected"


# Load data
data_path = Path("..")/"data"/"cybersecurity_intrusion_data.csv"
intrusion_df = pd.read_csv(data_path, keep_default_na=False)

mkdir -p failed for path /home/user/.cache/matplotlib: [Errno 13] Permission denied: '/home/user/.cache/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-wys297qg because there was an issue with the default path (/home/user/.cache/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [4]:
intrusion_df

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0
...,...,...,...,...,...,...,...,...,...,...,...
9532,SID_09533,194,ICMP,3,226.049889,AES,0.517737,3,Chrome,0,1
9533,SID_09534,380,TCP,3,182.848475,None,0.408485,0,Chrome,0,0
9534,SID_09535,664,TCP,5,35.170248,AES,0.359200,1,Firefox,0,0
9535,SID_09536,406,TCP,4,86.664703,AES,0.537417,1,Chrome,1,0


In [5]:
intrusion_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           9537 non-null   object 
 1   network_packet_size  9537 non-null   int64  
 2   protocol_type        9537 non-null   object 
 3   login_attempts       9537 non-null   int64  
 4   session_duration     9537 non-null   float64
 5   encryption_used      9537 non-null   object 
 6   ip_reputation_score  9537 non-null   float64
 7   failed_logins        9537 non-null   int64  
 8   browser_type         9537 non-null   object 
 9   unusual_time_access  9537 non-null   int64  
 10  attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 819.7+ KB


In [24]:
intrusion_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
session_id,9537,9537,SID_00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
network_packet_size,9537.0,NaN,NaN,NaN,500.430639,198.379364,64.0,365.0,499.0,635.0,1285.0
protocol_type,9537,3,TCP,6624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
login_attempts,9537.0,NaN,NaN,NaN,4.032086,1.963012,1.0,3.0,4.0,5.0,13.0
session_duration,9537.0,NaN,NaN,NaN,792.745312,786.560144,0.5,231.953006,556.277457,1105.380602,7190.392213
encryption_used,9537,3,AES,4706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_reputation_score,9537.0,NaN,NaN,NaN,0.331338,0.177175,0.002497,0.191946,0.314778,0.453388,0.924299
failed_logins,9537.0,NaN,NaN,NaN,1.517773,1.033988,0.0,1.0,1.0,2.0,5.0
browser_type,9537,5,Chrome,5137,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unusual_time_access,9537.0,NaN,NaN,NaN,0.149942,0.357034,0.0,0.0,0.0,0.0,1.0


In [6]:
print("Missing values per column:\n", intrusion_df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", intrusion_df.duplicated().sum())
print()
intrusion_df["attack_detected"].value_counts()

Missing values per column:
 session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64

Duplicate rows: 0



attack_detected
0    5273
1    4264
Name: count, dtype: int64

# 2) Cleaning data

In [7]:
# Drop session id column
intrusion_df = intrusion_df.drop(columns=["session_id"])
intrusion_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   network_packet_size  9537 non-null   int64  
 1   protocol_type        9537 non-null   object 
 2   login_attempts       9537 non-null   int64  
 3   session_duration     9537 non-null   float64
 4   encryption_used      9537 non-null   object 
 5   ip_reputation_score  9537 non-null   float64
 6   failed_logins        9537 non-null   int64  
 7   browser_type         9537 non-null   object 
 8   unusual_time_access  9537 non-null   int64  
 9   attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 745.2+ KB


# 3) Train and Split data

In [8]:
X = intrusion_df.drop(columns=[TARGET_COLUMN]).copy()
y = intrusion_df[TARGET_COLUMN].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)


Train shape: (7629, 9)
Test shape:  (1908, 9)

Numeric features: ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access']
Categorical features: ['protocol_type', 'encryption_used', 'browser_type']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),]), numeric_features,),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore")),]), categorical_features,),
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [14]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
models = {
    "Perceptron": Perceptron(random_state=RANDOM_STATE, max_iter=2000, tol=1e-3),
    "LogisticRegression": LogisticRegression(random_state=RANDOM_STATE, max_iter=3000),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=400, min_samples_leaf=2, n_jobs=1,),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost" : XGBClassifier(random_state=RANDOM_STATE, objective="binary:logistic", eval_metric="logloss", tree_method="hist", n_estimators=400, learning_rate=0.05, max_depth=6, min_child_weight=1, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, n_jobs=1,)
}
print("Models to compare:", list(models.keys()))

Models to compare: ['Perceptron', 'LogisticRegression', 'KNN', 'RandomForest', 'GradientBoosting', 'HistGradientBoosting', 'XGBoost']


In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

model_pipelines = {}
cv_rows = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    model_pipelines[model_name] = pipeline

    cv_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1,)

    row = {"model": model_name}
    for metric_name in scoring:
        row[f"{metric_name}_mean"] = cv_scores[f"test_{metric_name}"].mean()
        row[f"{metric_name}_std"] = cv_scores[f"test_{metric_name}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values(by="f1_mean", ascending=False).reset_index(drop=True)
cv_results.round(4)


,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RandomForest,0.8920,0.0067,0.9985,0.0022,0.7596,0.0152,0.8627,0.0098,0.8842,0.0097
1,GradientBoosting,0.8916,0.0064,0.9973,0.0023,0.7596,0.0143,0.8623,0.0093,0.8830,0.0116
2,HistGradientBoosting,0.8879,0.0057,0.9852,0.0041,0.7608,0.0137,0.8585,0.0084,0.8854,0.0095
3,XGBoost,0.8860,0.0057,0.9764,0.0043,0.7634,0.0140,0.8568,0.0084,0.8856,0.0096
4,KNN,0.8230,0.0133,0.9449,0.0090,0.6415,0.0252,0.7640,0.0205,0.8655,0.0083
5,LogisticRegression,0.7411,0.0082,0.7280,0.0090,0.6722,0.0192,0.6989,0.0122,0.8038,0.0145
6,Perceptron,0.6322,0.0220,0.5909,0.0444,0.6213,0.1005,0.5983,0.0367,0.6926,0.0269


# Increase Recall

In [18]:
best_model = models["RandomForest"]

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model),
])

final_pipeline.fit(X_train, y_train)

y_probs = final_pipeline.predict_proba(X_test)[:, 1]

# Apply custom threshold (BOOST RECALL)
y_pred = (y_probs > 0.30).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.92      0.88      1055
           1       0.88      0.80      0.84       853

    accuracy                           0.86      1908
   macro avg       0.87      0.86      0.86      1908
weighted avg       0.87      0.86      0.86      1908

Confusion Matrix:

[[966  89]
 [169 684]]
